# Lofi Generator

Using `"saikayala/jazz-ml-ready-midi"` from `kaggle` for training (more jazzy sounding).

## Prereqs

Let's get everything setup, first.

In [ ]:
!apt-get update -qq && apt-get install -y lilypond > /dev/null

In [ ]:
%pip install -q music21 kagglehub tqdm torch pydantic

In [ ]:
from music21 import environment

try:
    env = environment.UserSettings()
    env["lilypondPath"] = "/usr/bin/lilypond"
    env["graphicsMagickPath"] = "/usr/bin/display"
except Exception as e:
    print(f"Setup warning: {e}")

## Pydantic config

Will be helpful for type hinting, etc.

In [ ]:
from pydantic import BaseModel, Field
from typing import Optional
import torch


class MusicConfig(BaseModel):
    dataset_handle: str = "saikayala/jazz-ml-ready-midi"
    extract_path: str = "./jazz_midi_extracted"

    sequence_length: int = 64
    embedding_dim: int = 256
    hidden_dim: int = 512
    num_layers: int = 2
    dropout: float = 0.3

    batch_size: int = 64
    learning_rate: float = 0.001
    epochs: int = 20
    device: str = "cuda" if torch.cuda.is_available() else "cpu"

## Preprocessing

Turning MIDI files into tokens (key, time, notes).

In [ ]:
import glob
import kagglehub
import os
import tarfile
from music21 import converter, instrument, note, chord, score, key
from tqdm.notebook import tqdm


class JazzPreprocessor:
    def __init__(self, config: MusicConfig):
        self.config = config
        self.vocab = []
        self.token_to_int = {}
        self.int_to_token = {}

    def setup_data(self):
        path = kagglehub.dataset_download(self.config.dataset_handle)
        archive_file = os.path.join(path, "Jazz-Midi.tar.xz")

        if not os.path.exists(self.config.extract_path):
            os.makedirs(self.config.extract_path)
            with tarfile.open(archive_file, "r:xz") as tar:
                for member in tqdm(tar.getmembers(), desc="Unpacking MIDI"):
                    tar.extract(member, path=self.config.extract_path)

        return glob.glob(
            os.path.join(self.config.extract_path, "**/*.mid"), recursive=True
        )

    def extract_metadata(self, score):
        try:
            k = score.analyze("key")
            key_token = f"KEY_{k.tonic.name}_{k.mode}"
            ts = score.getTimeSignatures()[0]
            ts_token = f"TIME_{ts.numerator}_{ts.denominator}"

            return [key_token, ts_token]
        except:
            return ["KEY_C_major", "TIME_4_4"]

    def tokenize_midi(self, file_path):
        try:
            score = converter.parse(file_path)
            meta = self.extract_metadata(score)
            tokens = meta

            parts = instrument.partitionByInstrument(score)
            tracks = parts.parts if parts else [score]

            for part in tracks:
                inst = part.partName if part.partName else "Piano"
                tokens.append(f"START_{inst}")
                for el in part.flatten().notesAndRests:
                    if isinstance(el, note.Note):
                        tokens.append(f"n_{el.pitch.nameWithOctave}")
                    elif isinstance(el, chord.Chord):
                        p_str = ".".join(sorted([n.nameWithOctave for n in el.pitches]))
                        tokens.append(f"c_{p_str}")
                    elif isinstance(el, note.Rest):
                        tokens.append("r")
                tokens.append(f"END_{inst}")
            return tokens
        except:
            return None

## Token to MIDI Decoder

This will have `lilypond` support.

In [ ]:
class MidiConverter:
    @staticmethod
    def tokens_to_score(tokens: list[str], quarter_length: float = 0.5):
        score = stream.Score()
        current_part = None
        global_key, global_ts = None, None

        for t in tokens:
            if t.startswith("KEY_"):
                p = t.split("_")
                global_key = key.Key(p[1], p[2])
            elif t.startswith("TIME_"):
                p = t.split("_")
                global_ts = meter.TimeSignature(f"{p[1]}/{p[2]}")
            elif t.startswith("START_"):
                inst_name = t.replace("START_", "")
                current_part = stream.Part()
                try:
                    inst_obj = getattr(instrument, inst_name)()
                except:
                    inst_obj = instrument.Piano()
                current_part.insert(0, inst_obj)
                if global_key:
                    current_part.insert(0, global_key)
                if global_ts:
                    current_part.insert(0, global_ts)
            elif t.startswith("END_"):
                if current_part:
                    score.insert(0, current_part)
                    current_part = None
            elif current_part is not None:
                if t.startswith("n_"):
                    n = note.Note(t.replace("n_", ""))
                    n.duration.quarterLength = quarter_length
                    current_part.append(n)
                elif t.startswith("c_"):
                    pitches = t.replace("c_", "").split(".")
                    c = chord.Chord(pitches)
                    c.duration.quarterLength = quarter_length * 2
                    current_part.append(c)
                elif t == "r":
                    r = note.Rest()
                    r.duration.quarterLength = quarter_length
                    current_part.append(r)

        if current_part:
            score.insert(0, current_part)

        return score

In [ ]:
def show_score(tokens, play=False):
    score = MidiConverter.tokens_to_score(tokens)
    if play:
        score.show("midi")
    else:
        score.show("lily.pdf")